<a href="https://colab.research.google.com/github/MBR4V0/Python/blob/main/Teste_game1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from IPython.display import HTML, display

html = r'''
<div style="text-align:center;font-family:sans-serif">
  <canvas id="g" width="800" height="480" tabindex="0"
    style="width:100%;max-width:800px;border:3px solid #333;border-radius:8px;outline:none;background:#87ceeb"></canvas>
  <p style="margin:6px;color:#555">Clique no jogo para focar. ← → / A D = mover | ↑ / W / ESPAÇO = pular (pule em cima dos inimigos!) | ENTER = reiniciar</p>
</div>
<script>
(function(){
const cv=document.getElementById('g'),ctx=cv.getContext('2d');
const T=32,W=800,H=480,ROWS=15;
let keys={},state='play',level=1,lives=3,score=0,coinsGot=0,frame=0,timer=0;
let grid,cols,enemies,coins,p,camX=0,jumpHeld=false,coyote=0,buf=0;
let audio=null;

cv.addEventListener('click',()=>cv.focus());
addEventListener('keydown',e=>{
  keys[e.code]=true;
  if(['ArrowUp','ArrowDown','ArrowLeft','ArrowRight','Space'].includes(e.code))e.preventDefault();
  if(!audio){try{audio=new (window.AudioContext||window.webkitAudioContext)();}catch(_){}}
});
addEventListener('keyup',e=>{keys[e.code]=false;});

function beep(f,d,type){
  if(!audio)return;
  try{
    const o=audio.createOscillator(),g=audio.createGain();
    o.type=type||'square';o.frequency.value=f;g.gain.value=0.05;
    o.connect(g);g.connect(audio.destination);
    o.start();g.gain.exponentialRampToValueAtTime(0.0001,audio.currentTime+d);
    o.stop(audio.currentTime+d);
  }catch(_){}
}

function rng(seed){return function(){seed|=0;seed=seed+0x6D2B79F5|0;let t=Math.imul(seed^seed>>>15,1|seed);t=t+Math.imul(t^t>>>7,61|t)^t;return((t^t>>>14)>>>0)/4294967296;}}

function loadLevel(){
  const r=rng(1234+level*77);
  cols=110+level*15;
  grid=Array.from({length:ROWS},()=>Array(cols).fill(0));
  enemies=[];coins=[];
  const gap=new Array(cols).fill(false);
  let x=12;
  while(x<cols-12){
    if(r()<0.65){const w=2+Math.floor(r()*2);for(let i=0;i<w;i++)gap[x+i]=true;x+=w;}
    x+=5+Math.floor(r()*5);
  }
  for(let c=0;c<cols;c++)if(!gap[c]){grid[13][c]=1;grid[14][c]=3;}
  // plataformas flutuantes
  let c=10;
  while(c<cols-14){
    const len=3+Math.floor(r()*3),row=10+Math.floor(r()*2);
    for(let i=0;i<len;i++)grid[row][c+i]=2;
    for(let i=0;i<len;i++)if(r()<0.8)coins.push({x:(c+i)*T+8,y:(row-1)*T+6,got:false});
    c+=len+5+Math.floor(r()*6);
  }
  // blocos empilhados
  for(let c=8;c<cols-10;c++){
    let ok=true;
    for(let d=-2;d<=2;d++){
      if(gap[c+d]||grid[10][c+d]||grid[11][c+d]||grid[12][c+d])ok=false;
    }
    if(ok&&r()<0.07){
      const h=1+Math.floor(r()*2);
      for(let i=0;i<h;i++)grid[12-i][c]=2;
      c+=3;
    }
  }
  // moedas no chão
  for(let c=14;c<cols-10;c++)if(!gap[c]&&grid[12][c]===0&&r()<0.06)coins.push({x:c*T+8,y:11*T+6,got:false});
  // inimigos
  const prob=Math.min(0.12,0.05+level*0.015);
  for(let c=16;c<cols-10;c++){
    if(grid[13][c]===1&&grid[12][c]===0&&grid[12][c-1]===0&&grid[12][c+1]===0&&r()<prob){
      enemies.push({x:c*T+3,y:13*T-24,w:26,h:24,vx:0,vy:0,dir:r()<0.5?-1:1,dead:false,t:0,onGround:false});
      c+=2;
    }
  }
  p={x:2*T,y:13*T-29,w:24,h:28,vx:0,vy:0,dir:1,onGround:false,inv:60};
  camX=0;state='play';
}

function newGame(){level=1;lives=3;score=0;coinsGot=0;loadLevel();}

function solid(tx,ty){
  if(tx<0||tx>=cols)return true;
  if(ty<0||ty>=ROWS)return false;
  return grid[ty][tx]>0;
}
function moveX(e){
  e.x+=e.vx;
  const y0=Math.floor(e.y/T),y1=Math.floor((e.y+e.h-0.01)/T);
  const x0=Math.floor(e.x/T),x1=Math.floor((e.x+e.w-0.01)/T);
  for(let ty=y0;ty<=y1;ty++)for(let tx=x0;tx<=x1;tx++){
    if(solid(tx,ty)){
      if(e.vx>0)e.x=tx*T-e.w;else if(e.vx<0)e.x=(tx+1)*T;
      return true;
    }
  }
  return false;
}
function moveY(e){
  e.y+=e.vy;e.onGround=false;
  const y0=Math.floor(e.y/T),y1=Math.floor((e.y+e.h-0.01)/T);
  const x0=Math.floor(e.x/T),x1=Math.floor((e.x+e.w-0.01)/T);
  for(let ty=y0;ty<=y1;ty++)for(let tx=x0;tx<=x1;tx++){
    if(solid(tx,ty)){
      if(e.vy>0){e.y=ty*T-e.h;e.onGround=true;}else if(e.vy<0){e.y=(ty+1)*T;}
      e.vy=0;return;
    }
  }
}

function die(){
  lives--;beep(150,0.4,'sawtooth');
  if(lives<=0){state='over';return;}
  p.x=Math.max(2*T,p.x-12*T);p.y=0;p.vx=0;p.vy=0;p.inv=120;
  // procura chão seguro
  let tx=Math.floor(p.x/T);
  while(tx>2&&!(grid[13][tx]>0&&grid[13][tx+1]>0&&grid[13][tx-1]>0))tx--;
  p.x=tx*T+4;p.y=0;
}

function updatePlayer(){
  const L=keys.ArrowLeft||keys.KeyA,R=keys.ArrowRight||keys.KeyD;
  const J=keys.ArrowUp||keys.KeyW||keys.Space;
  if(L){p.vx-=0.5;p.dir=-1;}
  if(R){p.vx+=0.5;p.dir=1;}
  if(!L&&!R)p.vx*=0.8;
  p.vx=Math.max(-4.5,Math.min(4.5,p.vx));
  if(Math.abs(p.vx)<0.05)p.vx=0;
  if(p.onGround)coyote=6;else coyote--;
  if(J&&!jumpHeld)buf=6;else buf--;
  if(buf>0&&coyote>0){p.vy=-12;coyote=0;buf=0;beep(520,0.12);}
  if(!J&&p.vy<-4)p.vy=-4;
  jumpHeld=!!J;
  p.vy=Math.min(14,p.vy+0.6);
  if(moveX(p))p.vx=0;
  moveY(p);
  if(p.x<0)p.x=0;
  if(p.inv>0)p.inv--;
  if(p.y>H+80)die();
}

function update(){
  frame++;
  if(state==='over'){if(keys.Enter)newGame();return;}
  if(state==='levelup'){timer--;if(timer<=0){level++;loadLevel();}return;}
  updatePlayer();
  for(const e of enemies){
    if(e.dead){e.t--;continue;}
    e.vy=Math.min(14,e.vy+0.6);
    e.vx=e.dir*(1+level*0.1);
    if(moveX(e))e.dir*=-1;
    moveY(e);
    if(e.onGround){
      const fx=e.dir>0?e.x+e.w+2:e.x-2;
      if(!solid(Math.floor(fx/T),Math.floor((e.y+e.h+4)/T)))e.dir*=-1;
    }
    if(e.y>H+80){e.dead=true;e.t=0;}
    // colisão com o jogador
    if(!e.dead&&p.x<e.x+e.w&&p.x+p.w>e.x&&p.y<e.y+e.h&&p.y+p.h>e.y){
      if(p.vy>0&&(p.y+p.h-e.y)<20){
        e.dead=true;e.t=20;p.vy=-8;score+=100;beep(700,0.15);
      }else if(p.inv<=0){die();}
    }
  }
  enemies=enemies.filter(e=>!e.dead||e.t>0);
  for(const c of coins){
    if(!c.got&&p.x<c.x+16&&p.x+p.w>c.x&&p.y<c.y+20&&p.y+p.h>c.y){
      c.got=true;coinsGot++;score+=10;beep(1000,0.1,'triangle');
    }
  }
  // bandeira
  const poleX=(cols-6)*T+14;
  if(p.x+p.w>poleX&&state==='play'){state='levelup';timer=120;score+=500;beep(880,0.4,'triangle');}
  camX=Math.max(0,Math.min(cols*T-W,p.x-W/3));
}

// ---------- DESENHO ----------
function drawBG(){
  const g=ctx.createLinearGradient(0,0,0,H);
  g.addColorStop(0,'#5fb3ff');g.addColorStop(1,'#d6f0ff');
  ctx.fillStyle=g;ctx.fillRect(0,0,W,H);
  ctx.fillStyle='#9ad98a';
  ctx.beginPath();ctx.moveTo(0,H);
  for(let x=0;x<=W;x+=10){
    ctx.lineTo(x,330+Math.sin((x+camX*0.3)/90)*30+Math.sin((x+camX*0.3)/37)*8);
  }
  ctx.lineTo(W,H);ctx.fill();
  ctx.fillStyle='rgba(255,255,255,0.9)';
  for(let i=0;i<30;i++){
    const wx=i*300+(i*71)%100;
    let x=((wx-camX*0.4)%(W+300)+(W+300))%(W+300)-150;
    const y=40+(i*37)%90;
    ctx.beginPath();
    ctx.arc(x,y,20,0,7);ctx.arc(x+22,y-8,24,0,7);ctx.arc(x+48,y,20,0,7);ctx.fill();
  }
}
function drawTiles(){
  const c0=Math.floor(camX/T),c1=Math.min(cols-1,c0+Math.ceil(W/T)+1);
  for(let r=0;r<ROWS;r++)for(let c=c0;c<=c1;c++){
    const t=grid[r][c];if(!t)continue;
    const x=c*T-camX,y=r*T;
    if(t===2){
      ctx.fillStyle='#d9822b';ctx.fillRect(x,y,T,T);
      ctx.strokeStyle='#8a4b12';ctx.lineWidth=2;ctx.strokeRect(x+1,y+1,T-2,T-2);
      ctx.beginPath();ctx.moveTo(x,y+T/2);ctx.lineTo(x+T,y+T/2);
      ctx.moveTo(x+T/2,y);ctx.lineTo(x+T/2,y+T/2);ctx.stroke();
    }else{
      ctx.fillStyle='#8b5a2b';ctx.fillRect(x,y,T,T);
      ctx.fillStyle='#74481f';ctx.fillRect(x+6,y+10,5,5);ctx.fillRect(x+20,y+20,5,5);
      if(t===1){ctx.fillStyle='#4cc04c';ctx.fillRect(x,y,T,9);ctx.fillStyle='#2e9a2e';ctx.fillRect(x,y+8,T,2);}
    }
  }
}
function drawCoins(){
  for(const c of coins){
    if(c.got)continue;
    const w=Math.abs(Math.cos((frame+c.x)/12))*7+2;
    ctx.fillStyle='#ffd700';ctx.strokeStyle='#b8860b';ctx.lineWidth=2;
    ctx.beginPath();ctx.ellipse(c.x+8-camX,c.y+10,w,9,0,0,7);ctx.fill();ctx.stroke();
  }
}
function drawFlag(){
  const x=(cols-6)*T+14-camX;
  ctx.fillStyle='#eee';ctx.fillRect(x,6*T,5,7*T);
  ctx.fillStyle='#e63946';
  ctx.beginPath();ctx.moveTo(x+5,6*T);ctx.lineTo(x+50,6*T+16+Math.sin(frame/8)*3);ctx.lineTo(x+5,6*T+32);ctx.fill();
  ctx.fillStyle='#ffd700';ctx.beginPath();ctx.arc(x+2,6*T-4,6,0,7);ctx.fill();
}
function drawEnemy(e){
  const x=e.x-camX,y=e.y;
  if(e.dead){
    ctx.fillStyle='#7a3b8f';ctx.fillRect(x,y+e.h-6,e.w,6);return;
  }
  ctx.fillStyle='#8e44ad';
  ctx.beginPath();ctx.ellipse(x+e.w/2,y+e.h/2,e.w/2,e.h/2,0,0,7);ctx.fill();
  ctx.fillStyle='#5b2c6f';
  const s=Math.sin(frame/5)*3;
  ctx.fillRect(x+2+s,y+e.h-4,8,4);ctx.fillRect(x+e.w-10-s,y+e.h-4,8,4);
  ctx.fillStyle='#fff';ctx.fillRect(x+5,y+6,6,7);ctx.fillRect(x+15,y+6,6,7);
  ctx.fillStyle='#000';ctx.fillRect(x+7+e.dir*2,y+8,3,4);ctx.fillRect(x+17+e.dir*2,y+8,3,4);
  ctx.strokeStyle='#000';ctx.lineWidth=2;
  ctx.beginPath();ctx.moveTo(x+4,y+4);ctx.lineTo(x+11,y+7);ctx.moveTo(x+22,y+4);ctx.lineTo(x+15,y+7);ctx.stroke();
}
function drawDuck(){
  if(p.inv>0&&Math.floor(frame/4)%2===0)return;
  ctx.save();
  ctx.translate(p.x-camX+p.w/2,p.y+p.h);
  ctx.scale(p.dir,1);
  const walk=p.onGround&&Math.abs(p.vx)>0.5?Math.sin(frame/3)*4:0;
  // pés
  ctx.fillStyle='#ff8c1a';
  ctx.fillRect(-9+walk,-5,9,5);ctx.fillRect(1-walk,-5,9,5);
  // corpo
  ctx.fillStyle='#ffe033';
  ctx.beginPath();ctx.ellipse(-1,-14,12,10,0,0,7);ctx.fill();
  // asa
  ctx.fillStyle='#f2c200';
  ctx.beginPath();ctx.ellipse(-5,-13,6,5,0.3,0,7);ctx.fill();
  // cabeça
  ctx.fillStyle='#ffe033';
  ctx.beginPath();ctx.arc(5,-24,8,0,7);ctx.fill();
  // bico
  ctx.fillStyle='#ff8c1a';
  ctx.beginPath();ctx.ellipse(14,-22,6,3,0,0,7);ctx.fill();
  // olho
  ctx.fillStyle='#fff';ctx.beginPath();ctx.arc(7,-26,3,0,7);ctx.fill();
  ctx.fillStyle='#000';ctx.beginPath();ctx.arc(8,-26,1.5,0,7);ctx.fill();
  // boné vermelho (estilo Mario)
  ctx.fillStyle='#e63946';
  ctx.beginPath();ctx.arc(4,-28,7,Math.PI,0);ctx.fill();
  ctx.fillRect(4,-29,10,3);
  ctx.restore();
}
function drawHUD(){
  ctx.fillStyle='rgba(0,0,0,0.45)';ctx.fillRect(0,0,W,30);
  ctx.fillStyle='#fff';ctx.font='bold 16px sans-serif';ctx.textAlign='left';
  ctx.fillText('Vidas: '+'\u2665'.repeat(Math.max(0,lives)),10,21);
  ctx.fillText('Moedas: '+coinsGot,170,21);
  ctx.fillText('Pontos: '+score,300,21);
  ctx.fillText('Fase: '+level,450,21);
  const prog=Math.min(1,p.x/((cols-6)*T));
  ctx.fillStyle='#555';ctx.fillRect(560,10,220,10);
  ctx.fillStyle='#4cc04c';ctx.fillRect(560,10,220*prog,10);
}
function center(t,y,size,color){
  ctx.textAlign='center';ctx.font='bold '+size+'px sans-serif';
  ctx.fillStyle='rgba(0,0,0,0.6)';ctx.fillText(t,W/2+2,y+2);
  ctx.fillStyle=color;ctx.fillText(t,W/2,y);
}
function draw(){
  drawBG();drawTiles();drawCoins();drawFlag();
  for(const e of enemies)drawEnemy(e);
  drawDuck();drawHUD();
  if(state==='levelup'){center('FASE COMPLETA!',230,48,'#ffd700');}
  if(state==='over'){
    ctx.fillStyle='rgba(0,0,0,0.6)';ctx.fillRect(0,0,W,H);
    center('GAME OVER',210,56,'#ff4d4d');
    center('Pontos: '+score+'  |  Moedas: '+coinsGot,260,24,'#fff');
    center('Pressione ENTER para jogar de novo',310,20,'#fff');
  }
}

let last=performance.now(),acc=0;
function loop(t){
  acc+=Math.min(100,t-last);last=t;
  while(acc>=1000/60){update();acc-=1000/60;}
  draw();requestAnimationFrame(loop);
}
newGame();
requestAnimationFrame(loop);
})();
</script>
'''
display(HTML(html))